# Agent2Agent Discovery And Invocation Localhost Demo

This notebook demonstrates the current InnomightLabs Agent2Agent (A2A) discovery and invocation flow:

1. Start from the configured InnomightLabs custom registry endpoint: `/a2a/agents`.
2. Read every registry item and populate its Agent Card from `agentCardUrl` or the embedded `agentCard` snapshot.
3. Parse each Agent Card with the official `a2a-sdk` parser.
4. Select an agent without hardcoded agent IDs or card paths.
5. Invoke the selected agent with the SDK `ClientFactory` and `SendMessageRequest`.

The only fixed service path in this notebook is the InnomightLabs registry path `/a2a/agents`. Agent Card URLs and A2A service URLs come from the registry/card payloads.



## Prerequisites

Start the API locally before running the notebook. For local development, the default API host is:

```text
http://localhost:1455
```

In the SPA, open an agent, create an API key if needed, and enable Agent2Agent Discovery on the agent overview page.

Install notebook dependencies in the kernel environment if they are not already available:

```bash
pip install a2a-sdk httpx
```

Do not commit live API keys into notebooks. Set the API key as an environment variable before launching Jupyter:

```bash
export A2A_API_KEY='<agent-api-key>'
```

Optional overrides:

```bash
export A2A_API_BASE_URL='http://localhost:1455'
export A2A_REGISTRY_URL='http://localhost:1455/a2a/agents'
export A2A_AGENT_ID='agent-id-to-select'
export A2A_AGENT_NAME='agent name to select'
export A2A_MESSAGE='Message to send to the selected agent'
```



In [ ]:
import json
import os
from urllib.parse import urlencode
from uuid import uuid4

import httpx
from a2a.client import ClientCallContext, ClientConfig, ClientFactory
from a2a.client.card_resolver import parse_agent_card
from a2a.types import SendMessageRequest
from a2a.utils.constants import TransportProtocol
from google.protobuf.json_format import MessageToDict, ParseDict

API_BASE_URL = os.getenv("A2A_API_BASE_URL", "http://localhost:1455").rstrip("/")
REGISTRY_URL = os.getenv("A2A_REGISTRY_URL", f"{API_BASE_URL}/a2a/agents").rstrip("/")
A2A_API_KEY = os.getenv("A2A_API_KEY", "").strip()
SELECT_AGENT_ID = os.getenv("A2A_AGENT_ID", "").strip()
SELECT_AGENT_NAME = os.getenv("A2A_AGENT_NAME", "").strip()
A2A_MESSAGE = os.getenv(
    "A2A_MESSAGE",
    "Hello from the Agent2Agent localhost notebook. Reply with one concise sentence confirming that A2A invocation is working.",
)

if not A2A_API_KEY:
    raise RuntimeError("Set A2A_API_KEY before running this notebook.")

print("API_BASE_URL:", API_BASE_URL)
print("REGISTRY_URL:", REGISTRY_URL)
print("A2A_API_KEY configured:", bool(A2A_API_KEY))
print("SELECT_AGENT_ID:", SELECT_AGENT_ID or "<first registry item>")
print("SELECT_AGENT_NAME:", SELECT_AGENT_NAME or "<not set>")



## Helper Functions

These helpers keep the notebook generic: card URLs are read from registry rows, service URLs are read from Agent Cards, and the SDK validates card shape.



In [ ]:
def print_json(payload: dict | list) -> None:
    print(json.dumps(payload, indent=2, sort_keys=False))


async def get_json_url(url: str, *, headers: dict[str, str] | None = None) -> dict:
    async with httpx.AsyncClient(timeout=120, follow_redirects=True) as client:
        response = await client.get(url, headers=headers)
        response.raise_for_status()
        payload = response.json()
    if not isinstance(payload, dict):
        raise RuntimeError(f"Expected JSON object from {url}")
    return payload


def bearer_headers() -> dict[str, str]:
    return {"Authorization": f"Bearer {A2A_API_KEY}"}


def sdk_parse_card(payload: dict) -> tuple[object, dict]:
    proto = parse_agent_card(payload)
    return proto, MessageToDict(proto)


def registry_card_url(item: dict) -> str:
    return str(item.get("agentCardUrl") or "").strip()


def registry_agent_id(item: dict) -> str:
    return str(item.get("id") or "").strip()


def preferred_interface(card: dict) -> dict:
    interfaces = card.get("supportedInterfaces") or []
    if not interfaces:
        raise RuntimeError("Agent Card is missing supportedInterfaces.")
    for interface in interfaces:
        if interface.get("protocolBinding") == "JSONRPC":
            return interface
    return interfaces[0]


def sdk_transport(protocol_binding: str) -> str:
    if protocol_binding == "JSONRPC":
        return TransportProtocol.JSONRPC.value
    if protocol_binding == "HTTP+JSON":
        return TransportProtocol.HTTP_JSON.value
    raise RuntimeError(f"Unsupported protocol binding: {protocol_binding}")



## Step 1: Fetch The Custom Registry

The registry URL is the configured discovery entrypoint. For InnomightLabs localhost it is `/a2a/agents` under the API base URL.



In [ ]:
registry_query_url = f"{REGISTRY_URL}?{urlencode({'limit': 100})}"
agent_listing = await get_json_url(registry_query_url)
print_json(agent_listing)



## Step 2: Populate Agent Cards From The Registry

Each registry item should provide `agentCardUrl` and may also include an embedded `agentCard` snapshot. The fetched `agentCardUrl` document is authoritative. This cell loads all Agent Cards found in the registry and parses them with `a2a-sdk`.



In [ ]:
async def load_registry_agent_cards(listing: dict) -> list[dict]:
    populated = []
    for item in listing.get("items", []):
        if not isinstance(item, dict):
            continue

        card_url = registry_card_url(item)
        embedded_card = item.get("agentCard") if isinstance(item.get("agentCard"), dict) else None
        if card_url:
            raw_card = await get_json_url(card_url)
            card_source = "agentCardUrl"
        elif embedded_card:
            raw_card = embedded_card
            card_source = "embedded agentCard"
        else:
            raise RuntimeError(f"Registry item is missing agentCardUrl and agentCard: {item}")

        card_proto, card = sdk_parse_card(raw_card)
        interface = preferred_interface(card)
        populated.append(
            {
                "id": registry_agent_id(item),
                "name": item.get("name") or card.get("name"),
                "description": item.get("description") or card.get("description"),
                "agentCardUrl": card_url or None,
                "cardSource": card_source,
                "serviceUrlFromCard": interface["url"],
                "protocolBinding": interface["protocolBinding"],
                "card": card,
                "cardProto": card_proto,
            }
        )
    return populated


discovered_agents = await load_registry_agent_cards(agent_listing)
if not discovered_agents:
    raise RuntimeError("No A2A-enabled agents found. Enable Agent2Agent Discovery for an agent first.")

print_json([
    {
        "id": agent["id"],
        "name": agent["name"],
        "agentCardUrl": agent["agentCardUrl"],
        "cardSource": agent["cardSource"],
        "serviceUrlFromCard": agent["serviceUrlFromCard"],
        "protocolBinding": agent["protocolBinding"],
        "skills": [skill.get("id") for skill in agent["card"].get("skills", [])],
    }
    for agent in discovered_agents
])



## Step 3: Select An Agent Without Hardcoded IDs

Set `A2A_AGENT_ID` or `A2A_AGENT_NAME` to choose a specific agent. If neither is set, the notebook uses the first registry item.



In [ ]:
def select_agent(agents: list[dict]) -> dict:
    if SELECT_AGENT_ID:
        match = next((agent for agent in agents if agent["id"] == SELECT_AGENT_ID), None)
        if not match:
            raise RuntimeError(f"A2A_AGENT_ID was not found in registry: {SELECT_AGENT_ID}")
        return match
    if SELECT_AGENT_NAME:
        needle = SELECT_AGENT_NAME.casefold()
        match = next((agent for agent in agents if str(agent["name"]).casefold() == needle), None)
        if not match:
            raise RuntimeError(f"A2A_AGENT_NAME was not found in registry: {SELECT_AGENT_NAME}")
        return match
    return agents[0]


selected_agent = select_agent(discovered_agents)
print_json({
    "id": selected_agent["id"],
    "name": selected_agent["name"],
    "description": selected_agent["description"],
    "agentCardUrl": selected_agent["agentCardUrl"],
    "serviceUrlFromCard": selected_agent["serviceUrlFromCard"],
    "protocolBindingFromCard": selected_agent["protocolBinding"],
})



## Step 4: Inspect The Selected Agent Card

The Agent Card is the A2A contract. The client should use `supportedInterfaces` from this card rather than constructing service URLs.



In [ ]:
print_json(selected_agent["card"])



## Step 5: Validate Public Data Is Sanitized

The public card should expose only public name, description, supported interfaces, auth scheme, modes, and public skills. It should not expose persona text, provider credentials, OAuth tokens, or installed skill secrets.



In [ ]:
public_payload = json.dumps(selected_agent["card"])

for forbidden in ["agent_persona", "agent_provider_api_key", "encrypted_credentials", "oauth", "secret", "apiKeySecurityScheme", "documentationUrl", "tenant"]:
    print(f"{forbidden!r} present:", forbidden in public_payload)



## Step 6: Show The Schema-Driven Skill Config Shape

The Agent2Agent Client skill now has a primary `registry_url` field plus optional `registry_urls` for additional discovery entries. At runtime the skill merges both fields and populates all Agent Cards found in each configured registry.



In [ ]:
skill_install_config_example = {
    "registry_set_name": "Local A2A Registry",
    "registry_url": REGISTRY_URL,
    "registry_urls": "",
    "default_credentials": {
        API_BASE_URL: "Bearer <agent api key>"
    },
}
print_json(skill_install_config_example)



## Step 7: Prepare Authentication

A2A invocation uses the agent API key as a bearer credential. The key is passed as an HTTP credential and is not included in the message body.



In [ ]:
auth_headers = bearer_headers()
redacted = f"{A2A_API_KEY[:8]}...{A2A_API_KEY[-4:]}"
print("Prepared Authorization header for API key:", redacted)
print_json({"Authorization": "Bearer <redacted>"})



## Step 8: Send A Message With The SDK

`ClientFactory` chooses the endpoint from the selected Agent Card. The notebook does not build the A2A service URL or JSON-RPC path manually.



In [ ]:
message_request = ParseDict(
    {
        "message": {
            "messageId": f"demo-message-{uuid4()}",
            "role": "ROLE_USER",
            "contextId": f"localhost-demo-{uuid4()}",
            "parts": [{"text": A2A_MESSAGE}],
        },
        "configuration": {
            "acceptedOutputModes": ["text/plain"],
        },
    },
    SendMessageRequest(),
    ignore_unknown_fields=False,
)

transport = sdk_transport(selected_agent["protocolBinding"])
async with httpx.AsyncClient(timeout=120, follow_redirects=True, headers=auth_headers) as sdk_http_client:
    sdk_client = ClientFactory(
        ClientConfig(
            streaming=False,
            httpx_client=sdk_http_client,
            supported_protocol_bindings=[transport],
            use_client_preference=True,
            accepted_output_modes=["text/plain"],
        )
    ).create(selected_agent["cardProto"])

    send_events = []
    async for event in sdk_client.send_message(
        message_request,
        context=ClientCallContext(timeout=120, service_parameters=auth_headers),
    ):
        send_events.append(MessageToDict(event))

if not send_events:
    raise RuntimeError("SDK send_message returned no events.")

send_response = send_events[-1]
print_json(send_response)



## Step 9: Extract The Agent Response Text

The SDK may return a `Task` or direct `Message`, depending on the remote agent implementation. This helper extracts text from either shape.



In [ ]:
def extract_message_text(message: dict | None) -> str | None:
    if not isinstance(message, dict):
        return None
    parts = message.get("parts") or []
    texts = [part.get("text") for part in parts if isinstance(part, dict) and part.get("text")]
    return "
".join(texts) if texts else None


def extract_response_text(response: dict) -> str | None:
    task = response.get("task") if isinstance(response.get("task"), dict) else None
    if task:
        status = task.get("status") if isinstance(task.get("status"), dict) else {}
        text = extract_message_text(status.get("message"))
        if text:
            return text
    return extract_message_text(response.get("message") if isinstance(response.get("message"), dict) else None)


response_text = extract_response_text(send_response)
print(response_text or "<no response text returned>")



## Expected Outcome

- The notebook starts at the custom InnomightLabs registry URL `/a2a/agents`.
- Registry rows use `id`, `agentCardUrl`, and embedded `agentCard` when present.
- Every discovered Agent Card is parsed by `a2a-sdk`.
- The selected service URL comes from `agentCard.supportedInterfaces`.
- `ClientFactory` sends `SendMessage` using the transport advertised by the selected Agent Card.
- No API key, agent ID, Agent Card path, or service URL is hardcoded in the notebook.

